In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Parámetros de uso común para la transpilación

<details>
<summary><b>Versiones de paquetes</b></summary>

El código de esta página fue desarrollado con los siguientes requisitos.
Se recomienda usar estas versiones o superiores.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Esta página describe algunos de los parámetros más utilizados para la transpilación local. Estos parámetros se configuran mediante argumentos de [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) o [`transpile`](https://docs.quantum.ibm.com/api/qiskit/compiler#qiskit.compiler.transpile).

<span id="approx-degree"></span>
## Grado de aproximación
Puedes usar el grado de aproximación para especificar qué tan cerca quieres que el circuito resultante coincida con el circuito deseado (de entrada). Es un número de punto flotante en el rango (0.0 - 1.0), donde 0.0 es la máxima aproximación y 1.0 (valor por defecto) significa sin aproximación. Los valores más bajos intercambian precisión en la salida por facilidad de ejecución (es decir, menos puertas). El valor predeterminado es 1.0.

En la síntesis unitaria de dos qubits (utilizada en las etapas iniciales de todos los niveles y en la etapa de optimización con nivel 3), este valor especifica la fidelidad objetivo de la descomposición de salida. Es decir, cuánto error se introduce cuando una representación matricial de un circuito se convierte a puertas discretas. Si el grado de aproximación es menor (más aproximación), el circuito de salida de la síntesis diferirá más de la matriz de entrada, pero probablemente tendrá menos puertas (ya que cualquier operación arbitraria de dos qubits puede descomponerse perfectamente con a lo sumo tres puertas CX) y será más fácil de ejecutar.

Cuando el grado de aproximación es menor que 1.0, pueden sintetizarse circuitos con una o dos puertas CX, lo que reduce el error proveniente del hardware pero aumenta el error por aproximación. Como CX es la puerta más costosa en términos de error, puede ser beneficioso reducir su cantidad a costa de la fidelidad en la síntesis (esta técnica se usó para aumentar el volumen cuántico en dispositivos IBM&reg;: [Validating quantum computers using randomized model circuits](https://arxiv.org/abs/1811.12926)).

Como ejemplo, generamos una `UnitaryGate` aleatoria de dos qubits que será sintetizada en la etapa inicial. Establecer `approximation_degree` en un valor menor que 1.0 puede generar un circuito aproximado. También debemos especificar `basis_gates` para que el método de síntesis sepa qué puertas puede usar en la síntesis aproximada.

In [1]:
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit.library import UnitaryGate
from qiskit.quantum_info import random_unitary
from qiskit.transpiler import generate_preset_pass_manager

UU = random_unitary(4, seed=12345)
rand_U = UnitaryGate(UU)

qubits = QuantumRegister(2, name="q")
qc = QuantumCircuit(qubits)
qc.append(rand_U, qubits)
pass_manager = generate_preset_pass_manager(
    optimization_level=1,
    approximation_degree=0.85,
    basis_gates=["sx", "rz", "cx"],
)
approx_qc = pass_manager.run(qc)
print(approx_qc.count_ops()["cx"])

2


This yields an output of `2` because the approximation requires fewer CX gates.

<span id="seed"></span>
## Random number generator seed

Some parts of the transpiler are stochastic, so repeated transpilation runs may return different results. To obtain a reproducible result, you can set the seed for the pseudorandom number generator using the `seed_transpiler` argument. Repeated runs using the same seed will return the same results.

Example:

In [2]:
pass_manager = generate_preset_pass_manager(
    optimization_level=1, seed_transpiler=11, basis_gates=["sx", "rz", "cx"]
)
optimized_1 = pass_manager.run(qc)
optimized_1.draw("mpl")

<Image src="../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg" alt="Output of the previous code cell" />

El resultado es `2` porque la aproximación requiere menos puertas CX.

<span id="seed"></span>
## Semilla del generador de números aleatorios
Algunas partes del transpilador son estocásticas, por lo que ejecuciones repetidas pueden devolver resultados distintos. Para obtener un resultado reproducible, puedes establecer la semilla del generador de números pseudoaleatorios con el argumento `seed_transpiler`. Las ejecuciones repetidas con la misma semilla siempre devuelven los mismos resultados.

Ejemplo:

In [3]:
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke
from qiskit.transpiler import Layout

backend = FakeSherbrooke()

a, b = qubits
initial_layout = Layout({a: 5, b: 6})

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/dbc652e8-53a4-47a9-a66e-d9c1e5ef07c9-0.svg)

<span id="init-layout"></span>
## Distribución inicial
Antes de la transpilación, los qubits de tu circuito son qubits virtuales que no necesariamente corresponden a qubits físicos del backend de destino. Puedes especificar la asignación inicial de qubits virtuales a qubits físicos mediante el argumento `initial_layout`. Ten en cuenta que la distribución final de qubits puede diferir de la inicial, ya que el transpilador puede permutar qubits usando puertas swap u otros mecanismos.

En el ejemplo siguiente, construimos una distribución inicial para el backend simulado [`FakeSherbrooke`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/fake-provider-fake-sherbrooke#fakesherbrooke) creando un objeto [`Layout`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.Layout). Nuestra distribución asigna el primer qubit del circuito al qubit 5 de Sherbrooke, y el segundo qubit del circuito al qubit 6 de Sherbrooke. Ten en cuenta que los qubits físicos siempre se representan con enteros.

In [4]:
initial_layout = [5, 6]

pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend, initial_layout=initial_layout
)
transpiled_circ = pass_manager.run(qc)

transpiled_circ.draw("mpl", idle_wires=False)

<Image src="../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/e18c034c-eb26-4d9d-81d7-37e0eafa17c7-0.svg)

Además de especificar un objeto Layout, también puedes pasar una lista de enteros, donde el $i$-ésimo elemento de la lista contiene el qubit físico al que debe asignarse el $i$-ésimo qubit. Por ejemplo:

In [5]:
from qiskit.visualization import plot_error_map

plot_error_map(backend, figsize=(30, 24))

<Image src="../docs/images/guides/common-parameters/extracted-outputs/8df57c6a-1ff4-4d58-9b7e-4378452c3025-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/common-parameters/extracted-outputs/a7800d8a-7354-48e4-a55f-f902ae28c875-0.svg)

Puedes usar la función [`plot_error_map`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.plot_error_map) para generar un diagrama del grafo del dispositivo con información de errores y con los qubits físicos etiquetados. También puedes ver diagramas similares en la página de [Recursos de cómputo](https://quantum.cloud.ibm.com/computers).